| Metric                      | What it measures                                 | How to compute                               | What you need                |
| --------------------------- | ------------------------------------------------ | -------------------------------------------- | ---------------------------- |
| **Numeric coverage MAE**    | Accuracy of cloud cover values (`hcc, mcc, lcc`) | Compare values extracted from text vs. CSV   | CSV + regex                  |
| **Cloud classification F1** | Correct presence/absence of high/mid/low clouds  | CSV >0.1 → present; search keywords in text  | CSV + keyword list           |
| **Variable coverage (%)**   | Whether all relevant layers are mentioned        | Count mentions / expected total              | Text + keywords              |
| **Logical contradictions**  | Detect phrases inconsistent with CSV             | Rules (e.g., “clear sky” with `tcc>0.8`)     | CSV + text                   |
| **CLIPScore**               | Alignment between description and image          | Compute cosine similarity of CLIP embeddings | Text + corresponding image   |
| **Output length**           | Verbosity                                        | Count tokens/words in description            | Text                         |
| **n-gram repetition**       | Fluency (avoid redundancy)                       | % of repeated bigrams/trigrams               | Text                         |
| **Readability**             | Ease of reading                                  | Automated readability index                  | Text                         |
| **Grammar errors**          | Basic correctness                                | Automated checker (e.g., LanguageTool)       | Text                         |
| **Inference time**          | Generation speed                                 | Measure `end_time - start_time`              | Timer during inference       |
| **Memory/CPU/GPU usage**    | Computational resource footprint                 | Use `psutil` or GPU logs                     | System monitor               |
| **Monetary cost**           | Training/inference cost                          | Tokens × price (API) or electricity × hours  | Billing data or estimation   |
| **Energy cost (kWh)**       | Environmental impact                             | Avg. power consumption × time                | Hardware meter or cloud logs |
| **Latency per case**        | User-facing responsiveness                       | Time per description                         | Timer during inference       |
| **Model size (GB)**         | Storage footprint                                | Model checkpoint or deployment size          | Model info                   |


Inference time → necesitarías guardar timestamps (start_time, end_time) en el momento de generación.

Memory/CPU/GPU usage → requiere medir en tiempo real (psutil, nvidia-smi o logs del cluster).

Monetary cost → requiere o bien datos de facturación (API tokens × precio) o coste eléctrico × horas.

Energy cost (kWh) → requiere logs de consumo eléctrico medio × tiempo.

Latency per case → lo mismo que inference time, guardando tiempos por ejemplo con un cronómetro.

Model size (GB) → metadatos del modelo usado (checkpoint, despliegue).

In [1]:
import os, re, json
import numpy as np, pandas as pd
from types import SimpleNamespace

In [2]:
# --------------------------- Config (fijo) ---------------------------
KW = {
    "high_pos":[r"\bhigh cloud(s)?\b",r"\bhigh[- ]altitude cloud(s)?\b",r"\bcirrus\b",r"\bcirro(?:stratus|cumulus)\b",r"\b(?:hcc|high cloud cover)\b"],
    "high_neg":[r"\bno (?:high[- ]altitude|high) cloud(s)?\b",r"\bwithout high cloud(s)?\b"],
    "mid_pos":[r"\bmid(?:dle)?[- ]level cloud(s)?\b",r"\baltostratus\b",r"\baltocumulus\b",r"\b(?:mcc|mid(?:dle)? cloud cover)\b"],
    "mid_neg":[r"\bno (?:mid(?:dle)?[- ]level|middle) cloud(s)?\b",r"\bwithout (?:mid|middle) cloud(s)?\b"],
    "low_pos":[r"\blow[- ]level cloud(s)?\b",r"\bstratus\b",r"\bstratocumulus\b",r"\bfog\b",r"\b(?:lcc|low cloud cover)\b"],
    "low_neg":[r"\bno (?:low[- ]level|low) cloud(s)?\b",r"\bwithout low cloud(s)?\b"],
    "clear":[r"\bclear sky\b",r"\bmostly clear\b",r"\bclear to fair\b",r"\bfair weather\b"],
    "overcast":[r"\bovercast\b",r"\b(?:nearly|almost) (?:the )?entire sky (?:is )?covered\b",r"\bsky (?:is )?dominated\b"],
}
NUM_PAT = {
    "hcc":[r"high cloud (?:cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",r"\bhcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?",r"high cloud(?:s)?[^%]*?(\d+(?:\.\d+)?)\s*%",r"\bnubes altas[^%]*?(\d+(?:\.\d+)?)\s*%"],
    "mcc":[r"(?:mid|middle|midlevel|mid-level) cloud (?:cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",r"\bmcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?",r"(?:mid|middle)[^%]*?(\d+(?:\.\d+)?)\s*%",r"\bnubes medias[^%]*?(\d+(?:\.\d+)?)\s*%"],
    "lcc":[r"(?:low|lowlevel|low-level) cloud (?:cover|coverage)\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%",r"\blcc\b\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?",r"(?:low)[^%]*?(\d+(?:\.\d+)?)\s*%",r"\bnubes bajas[^%]*?(\d+(?:\.\d+)?)\s*%"],
}

In [3]:
# --------------------------- Text utils ---------------------------
norm = lambda s: re.sub(r"\s+"," ",(s or "").strip().lower())
wtokens = lambda t: re.findall(r"[A-Za-zÀ-ÿ']+", (t or "").lower())
split_sents = lambda t: [s for s in re.split(r"[.!?]+",(t or "")) if s.strip()]

def any_match(pats, text): return any(re.search(p, text) for p in pats)

In [4]:
def presence_from_text(text, layer):
    t = norm(text)
    if layer=="high":
        return False if any_match(KW["high_neg"],t) else True if any_match(KW["high_pos"],t) else None
    if layer=="mid":
        return False if any_match(KW["mid_neg"],t) else True if any_match(KW["mid_pos"],t) else None
    if layer=="low":
        return False if any_match(KW["low_neg"],t) else True if any_match(KW["low_pos"],t) else None
    return None

In [5]:
def readability_proxy(text):
    s, w = split_sents(text), wtokens(text)
    if not s or not w: return np.nan
    avg_s, avg_w = len(w)/len(s), sum(map(len,w))/len(w)
    return max(0,min(100, 100 - (avg_s-15)*2 - (avg_w-5)*10))

In [6]:
def ngram_rep_ratio(text, n=2):
    toks = wtokens(text)
    if len(toks) < n+1: return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    counts = pd.Series(ngrams).value_counts()
    return float(((counts-1).clip(lower=0)).sum())/max(1,len(ngrams))

In [7]:
def extract_numeric(text, key):
    t = norm(text)
    for pat in NUM_PAT[key]:
        m = re.search(pat,t)
        if m:
            val=float(m.group(1))
            return val if 0<=val<=1 else val/100.0
    return None

In [8]:
# --------------------------- Metrics ---------------------------
def f1_from_conf(tp, fp, fn):
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return prec, rec, f1

In [9]:
def logical_contradictions_row(r, desc_col, tcc_col, hcc_col, mcc_col, lcc_col):
    d, tcc = norm(r.get(desc_col,"")), r.get(tcc_col, np.nan)
    c=0
    if not pd.isna(tcc):
        c += int(tcc>0.8 and any_match(KW["clear"],d))
        c += int(tcc<0.2 and any_match(KW["overcast"],d))
    flags = {
        "high": (r.get(hcc_col, np.nan)>0.1 if not pd.isna(r.get(hcc_col, np.nan)) else None),
        "mid":  (r.get(mcc_col, np.nan)>0.1 if not pd.isna(r.get(mcc_col, np.nan)) else None),
        "low":  (r.get(lcc_col, np.nan)>0.1 if not pd.isna(r.get(lcc_col, np.nan)) else None),
    }
    preds = {k:presence_from_text(d,k) for k in ["high","mid","low"]}
    for k in ["high","mid","low"]:
        if flags[k] is None or preds[k] is None: continue
        c += int(flags[k] and preds[k] is False) + int((not flags[k]) and preds[k] is True)
    return c

In [10]:
# --------------------- Optional: CLIPScore ---------------------
def compute_clip_scores(df, image_col, text_col, model="openai/clip-vit-base-patch32", batch=8, low=0.10, high=0.40, normalize=False, device=None):
    try:
        import torch; from transformers import CLIPProcessor, CLIPModel; from PIL import Image
    except Exception as e:
        print("[WARN] CLIP deps missing:",e); n=len(df); return [np.nan]*n, ([np.nan]*n if normalize else None)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    try:
        model = CLIPModel.from_pretrained(model).to(device).eval(); proc = CLIPProcessor.from_pretrained(model.name_or_path)
    except Exception as e:
        print("[WARN] CLIP load fail:",e); n=len(df); return [np.nan]*n, ([np.nan]*n if normalize else None)
    scores, norms = [], []
    with torch.inference_mode():
        for i in range(0,len(df),batch):
            b=df.iloc[i:i+batch]
            texts=b[text_col].fillna("").astype(str).tolist()
            imgs=[]
            for p in b[image_col].fillna("").astype(str):
                try: img=Image.open(p).convert("RGB")
                except Exception: img=Image.new("RGB",(224,224),(0,0,0))
                imgs.append(img)
            inp=proc(text=texts, images=imgs, return_tensors="pt", padding=True).to(device)
            out=model(**inp)
            ie, te = out.image_embeds, out.text_embeds
            cos=(te/te.norm(dim=-1,keepdim=True) * ie/ie.norm(dim=-1,keepdim=True)).sum(dim=-1).cpu().tolist()
            scores+=cos
            if normalize:
                s=np.clip(np.array(cos),low,high); s=(s-low)/(high-low)*100.0; norms+=s.tolist()
    return scores, (norms if normalize else None)

In [11]:
# ------------------ Optional: Grammar errors ------------------
def setup_language_tool(lang_code=None, use_spell=False):
    """
    Devuelve una función que cuenta 'errores' simples de estilo/gramática (proxy local).
    Mantiene la misma interfaz que language_tool_python.LanguageTool(...).
    """
    import re
    WORD=re.compile(r"[A-Za-zÀ-ÿ']+"); SENT=re.compile(r"[.!?]+"); REPW=re.compile(r"\b(\w+)\s+\1\b",re.I)
    BADP=re.compile(r"([!?.,;:])\1+"); BADSP=re.compile(r"([.!?,;:])(?!\s|$)"); LOWAF=re.compile(r"[.!?]\s+[a-záéíóúñ]")
    UPPER=re.compile(r"[A-ZÁÉÍÓÚÑ]"); URL=re.compile(r"https?://|www\.")
    sc=None
    if use_spell:
        try:
            from spellchecker import SpellChecker
            sc=SpellChecker(language=("es" if (lang_code or "").startswith("es") else "en"))
        except: sc=None

    def checker(text):
        t=(text or "").strip()
        if not t: return 0
        issues=0
        # Estructura y puntuación
        issues+=int("  " in t)+int(BADP.search(t) is not None)+int(BADSP.search(t) is not None)
        issues+=int(t.count("(")!=t.count(")"))+int(t.count("[")!=t.count("]"))+int(t.count("{")!=t.count("}"))
        issues+=int(t.count('"')%2)+int(t.count("'")%2 and not re.search(r"[A-Za-z]('[A-Za-z])",t))
        issues+=int("--" in t and not re.search(r"\s--\s",t))  # guiones mal espaciados
        # Frases
        sents=[s.strip() for s in SENT.split(t) if s.strip()]
        if sents:
            lens=[len(WORD.findall(s)) for s in sents]
            issues+=int(any(n>40 for n in lens))+int(any(n<4 for n in lens))   # muy largas / muy cortas
            issues+=int(not re.search(r"[.!?]\s*$",t))                         # sin punto final
            issues+=int(LOWAF.search(t) is not None)                           # minúscula tras punto
            issues+=sum(int(not re.match(r"^[A-ZÁÉÍÓÚÑ]",s)) for s in sents[:3])  # primeras frases sin mayúscula
        # Léxico/estilo
        toks=[w for w in WORD.findall(t)]
        if toks:
            cap_ratio=sum(1 for w in toks if UPPER.search(w) and w.isalpha())/max(1,len(toks))
            issues+=int(cap_ratio>0.2)+int(cap_ratio>0.35)                     # exceso de MAYÚSCULAS
        issues+=int(REPW.search(t) is not None)                                # palabra repetida
        issues+=int(URL.search(t) is not None)                                 # URLs en descripción
        # Comas potencialmente problemáticas (coma antes de 'and/y' + cláusulas largas)
        issues+=int(re.search(r",\s+(and|y)\s+\w{4,}",t,re.I) is not None)
        # Ortografía local opcional
        if sc and toks:
            unk=len(sc.unknown([w.lower() for w in toks])); r=unk/max(1,len(toks))
            issues+=int(r>0.05)+int(r>0.15)
        return issues
    return checker


In [12]:
# --------------------------- Pipeline ---------------------------
def ensure_cols(df, required):
    for c in required:
        if c not in df: df[c]=np.nan
    return df

In [13]:
# MAE por capa
def mae(true_col,pred_col, df):
    m=df[pred_col].notna() & df[true_col].notna()
    return (float(np.mean(np.abs(df.loc[m, true_col]-df.loc[m, pred_col]))) if m.sum() else np.nan, int(m.sum()))

In [14]:
def enrich(df, args):
    DESC, IMG = args.description_col, args.path_image_col
    # Constantes (no se configuran)
    HCC_COL, MCC_COL, LCC_COL, TCC_COL = "hcc","mcc","lcc","tcc"

    df = ensure_cols(df, [DESC,HCC_COL,MCC_COL,LCC_COL,TCC_COL,IMG])
    df["_desc"]=df[DESC].fillna("").astype(str)

    # Presencia por capa
    for layer in ["high","mid","low"]: df[f"mention_{layer}"]=df["_desc"].map(lambda t: presence_from_text(t,layer))
    df["variable_coverage_pct"]=df[[f"mention_{l}" for l in ["high","mid","low"]]].apply(lambda r: 100.0*sum(x is not None for x in r)/3.0,axis=1)

    # Métricas por capa
    layer_stats={}
    for layer,col in {"high":HCC_COL,"mid":MCC_COL,"low":LCC_COL}.items():
        truth=df[col].map(lambda x: (not pd.isna(x)) and (x>0.1))
        pred=df[f"mention_{layer}"]; m=pred.notna()
        tp=int(((pred[m]==True)&(truth[m]==True)).sum()); fp=int(((pred[m]==True)&(truth[m]==False)).sum()); fn=int(((pred[m]==False)&(truth[m]==True)).sum())
        prec,rec,f1=f1_from_conf(tp,fp,fn); layer_stats[layer]={"tp":tp,"fp":fp,"fn":fn,"n_eval":int(m.sum()),"precision":prec,"recall":rec,"f1":f1}
    macro_f1=float(np.mean([layer_stats[l]["f1"] for l in layer_stats]))

    # Por fila
    df["logical_contradictions_count"]=df.apply(lambda r: logical_contradictions_row(r, DESC, TCC_COL, HCC_COL, MCC_COL, LCC_COL),axis=1)
    df["word_count"]=df["_desc"].map(lambda t: len(wtokens(t)))
    df["bigram_repeat_ratio"]=df["_desc"].map(lambda t: ngram_rep_ratio(t,2))
    df["trigram_repeat_ratio"]=df["_desc"].map(lambda t: ngram_rep_ratio(t,3))
    df["readability_score_0_100"]=df["_desc"].map(readability_proxy)

    # Extracción numérica desde texto
    df[f"{HCC_COL}_pred_from_text"]=df["_desc"].map(lambda t: extract_numeric(t,"hcc"))
    df[f"{MCC_COL}_pred_from_text"]=df["_desc"].map(lambda t: extract_numeric(t,"mcc"))
    df[f"{LCC_COL}_pred_from_text"]=df["_desc"].map(lambda t: extract_numeric(t,"lcc"))

    mae_hcc, n_hcc = mae("hcc", "hcc_pred_from_text", df)
    mae_mcc, n_mcc = mae("mcc", "mcc_pred_from_text", df)
    mae_lcc, n_lcc = mae("lcc", "lcc_pred_from_text", df)

    # CLIP opcional
    df["clipscore"]=np.nan; df["clipscore_norm_0_100"]=np.nan
    if args.compute_clip and IMG in df and df[IMG].notna().any():
        sc, ns = compute_clip_scores(df, IMG, DESC, model=args.clip_model, batch=args.clip_batch_size, low=args.clip_norm_low, high=args.clip_norm_high, normalize=True)
        df["clipscore"]=sc; df["clipscore_norm_0_100"]=ns

    # Grammar opcional
    df["grammar_errors"]=np.nan
    if args.grammar_lang:
        fn=setup_language_tool(args.grammar_lang)
        if fn: df["grammar_errors"]=df["_desc"].map(fn)

    # Resumen
    summary = {
        "macro_f1_layers":macro_f1,
        **{f"f1_{k}":v["f1"] for k,v in layer_stats.items()},
        **{f"precision_{k}":v["precision"] for k,v in layer_stats.items()},
        **{f"recall_{k}":v["recall"] for k,v in layer_stats.items()},
        "avg_variable_coverage_pct":float(df["variable_coverage_pct"].mean()),
        "avg_logical_contradictions":float(df["logical_contradictions_count"].mean()),
        "avg_word_count":float(df["word_count"].mean()),
        "avg_bigram_repeat_ratio":float(df["bigram_repeat_ratio"].mean()),
        "avg_trigram_repeat_ratio":float(df["trigram_repeat_ratio"].mean()),
        "avg_readability_score_0_100":float(df["readability_score_0_100"].mean()),
        "mae_hcc":mae_hcc,"n_hcc_used":n_hcc,
        "mae_mcc":mae_mcc,"n_mcc_used":n_mcc,
        "mae_lcc":mae_lcc,"n_lcc_used":n_lcc,
        "avg_clipscore":float(pd.to_numeric(df["clipscore"],errors="coerce").mean()),
        "avg_clipscore_norm_0_100":float(pd.to_numeric(df["clipscore_norm_0_100"],errors="coerce").mean()),
        "avg_grammar_errors":float(pd.to_numeric(df["grammar_errors"],errors="coerce").mean()),
    }
    return df, summary

In [15]:
# --------------------------- MAIN (sin argumentos) ---------------------------
def main():
    # === ARCHIVOS ===
    input_csv = "../../resources/utils/images_weather_description_openAI.csv"
    out_dir   = "../../resources/utils/test_metrics_llm.csv"

    # === NOMBRES DE COLUMNAS (solo cambia si tu CSV usa otros) ===
    description_col = "description"
    path_image_col = "path_image"

    # === OPCIONES ===
    compute_clip = False
    clip_model = "openai/clip-vit-base-patch32"
    clip_norm_low, clip_norm_high = 0.10, 0.40
    clip_batch_size = 8
    grammar_lang = "en"

    # ---------- Carga y ejecución ----------
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(input_csv)

    args = SimpleNamespace(
        description_col=description_col,
        path_image_col=path_image_col,
        compute_clip=compute_clip,
        clip_model=clip_model,
        clip_norm_low=clip_norm_low,
        clip_norm_high=clip_norm_high,
        clip_batch_size=clip_batch_size,
        grammar_lang=grammar_lang,
    )

    df, summary = enrich(df, args)

    perrow_out = os.path.join(out_dir, "per_row_metrics.csv"); df.to_csv(perrow_out, index=False)
    sm_out = os.path.join(out_dir, "summary_metrics.json")
    with open(sm_out, "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

    print("== Summary ==")
    for k, v in summary.items(): print(f"{k}: {v}")
    print("\nWrote:\n -", perrow_out); print(" -", sm_out)

In [16]:
%%time

if __name__=="__main__": main()

== Summary ==
macro_f1_layers: 0.810673937279551
f1_high: 0.8033008658008658
f1_mid: 0.8430066603235015
f1_low: 0.7857142857142858
precision_high: 0.6712638480669229
precision_mid: 0.7286184210526315
precision_low: 0.6470588235294118
recall_high: 1.0
recall_mid: 1.0
recall_low: 1.0
avg_variable_coverage_pct: 26.69722142652535
avg_logical_contradictions: 0.3132340303637926
avg_word_count: 53.90217702663993
avg_bigram_repeat_ratio: 0.021882170433712066
avg_trigram_repeat_ratio: 0.0035365534274164083
avg_readability_score_0_100: 96.42445401757274
mae_hcc: 0.08824213612903227
n_hcc_used: 31
mae_mcc: 0.09200195000000001
n_mcc_used: 4
mae_lcc: 0.23683287679166665
n_lcc_used: 48
avg_clipscore: nan
avg_clipscore_norm_0_100: nan
avg_grammar_errors: 0.034803781151532515

Wrote:
 - ../../resources/utils/test_metrics_llm.csv\per_row_metrics.csv
 - ../../resources/utils/test_metrics_llm.csv\summary_metrics.json
CPU times: total: 9.62 s
Wall time: 9.85 s
